# UTS Big Data & Predictive Analytics Lanjut (ST153)
## Notebook 2 (Spark Cluster): Pemrosesan Terdistribusi RDD, SQL, dan Structured Streaming

**Prasyarat Sebelum Menjalankan Notebook Ini:**
1. **Spark Master (`192.168.100.72:7077`)** dan seluruh **PC Worker** berstatus `ALIVE` di Panel Lab Big Data.
2. Tombol **"Petakan share Z:"** sudah diklik baik di PC Master maupun di seluruh PC Worker.
3. File `Z:/wiki_snapshot.jsonl` sudah berhasil diekspor dari Notebook 1 (`01_MASTER_Kafka_Ingestion_dan_Export.ipynb`).

**Cakupan Soal:**
- **Soal 2 (Cell 1 - Cell 4):** Pemrosesan Spark RDD (*Accumulator*, `glom()`, *Narrow/Wide*, `reduceByKey` vs `groupByKey`, `cache()`, `toDebugString`).
- **Soal 3 (Cell 5 - Cell 8):** Pemrosesan Spark DataFrame & SQL (Skema eksplisit, 4 kueri analitik, deteksi lonjakan *window* 5 menit P99, `explain()`, optimasi *shuffle partitions*).
- **Soal 4 (Cell 9 - Cell 10):** Spark Structured Streaming (*Event time*, *window* 1 menit, *watermark* 2 menit, `foreachBatch`, `lastProgress`, dan uji *checkpoint fault-tolerance*).

In [1]:
# CELL 1: Inisialisasi SparkSession ke Spark Standalone Cluster (192.168.100.72:7077)
import json
import re
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, BooleanType
)
import pyspark.sql.functions as F

# Sertakan paket konektor Kafka untuk Spark 4.1.3 (Scala 2.13) agar siap dipakai hingga Soal 4
spark = (
    SparkSession.builder
    .appName("UTS_BigData_Cluster_ST153")
    .master("spark://192.168.110.109:7077")
    .config("spark.cores.max", "6")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Cluster Terhubung! Master:", sc.master)
print("Buka App UI (port 4040) di Panel Lab untuk mengambil screenshot Stages & SQL.")

Spark Cluster Terhubung! Master: spark://192.168.110.109:7077
Buka App UI (port 4040) di Panel Lab untuk mengambil screenshot Stages & SQL.


In [2]:
!pip install pyspark==4.1.3

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/455.5 MB ? eta -:--:--
     ---------------------------------------- 1.3/455.5 MB 9.8 MB/s eta 0:00:47
     ---------------------------------------- 2.6/455.5 MB 7.3 MB/s eta 0:01:02
     ---------------------------------------- 5.2/455.5 MB 9.7 MB/s eta 0:00:47
      -------------------------------------- 7.6/455.5 MB 10.1 MB/s eta 0:00:45
      ------------------------------------- 10.2/455.5 MB 10.5 MB/s eta 0:00:43
     - ------------------------------------ 12.6/455.5 MB 10.7 MB/s eta 0:00:42
     - ------------------------------------ 14.9/455.5 MB 10.9 MB/s eta 0:00:41
     - ------------------------------------ 17.3/455.5 MB 11.0 MB/s eta 0:00:41
     - ------------------------------------ 19.4/455.5 MB 11.0 MB/s eta 0:00:40
     - ------------------------------------ 22.0/455.5 MB 11.0 MB/s eta 0:00:40
     -- ----------------------------------- 24.4/


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2: SOAL 2a & 2d (Bagian 1) - Parsing JSON dengan Accumulator, glom(), dan Pengukuran cache()
bad_records_acc = sc.accumulator(0)

def parse_json_line(line):
    import json  # Wajib di dalam fungsi agar Worker mengenal modul json
    try:
        record = json.loads(line)
        if not isinstance(record, dict) or 'wiki' not in record:
            bad_records_acc.add(1)
            return []
        return [record]
    except Exception:
        bad_records_acc.add(1)
        return []

# Menggunakan nama file wikipedia_snapshot.jsonl di folder C:/BigData
with open("C:/BigData/wikipedia_snapshot.jsonl", "r", encoding="utf-8") as f:
    local_data = f.readlines()

raw_rdd = sc.parallelize(local_data, numSlices=8)
parsed_rdd = raw_rdd.flatMap(parse_json_line)

total_valid = parsed_rdd.count()
num_partitions = parsed_rdd.getNumPartitions()
partition_counts = parsed_rdd.glom().map(len).collect()

print("=" * 65)
print("SOAL 2a: HASIL PARSING & DISTRIBUSI PARTISI DI WORKER")
print(f"Total baris valid      : {total_valid}")
print(f"Total baris rusak (Acc): {bad_records_acc.value}")
print(f"Jumlah partisi         : {num_partitions}")
print(f"Sebaran per partisi    : {partition_counts}")
print("=" * 65)

# Pengukuran efek cache() sebelum dan sesudah (Soal 2d)
t0 = time.time()
_ = parsed_rdd.count()
t_no_cache_1 = time.time() - t0

t0 = time.time()
_ = parsed_rdd.filter(lambda x: x.get('wiki') == 'idwiki').count()
t_no_cache_2 = time.time() - t0
print(f"\n[Tanpa Cache] Action 1: {t_no_cache_1:.3f}s | Action 2: {t_no_cache_2:.3f}s")

parsed_rdd.cache()

t0 = time.time()
_ = parsed_rdd.count()  # Materialisasi ke memori Worker
t_cache_1 = time.time() - t0

t0 = time.time()
_ = parsed_rdd.filter(lambda x: x.get('wiki') == 'idwiki').count()  # Baca langsung dari RAM Worker
t_cache_2 = time.time() - t0
print(f"[Dengan Cache] Action 1 (materialisasi): {t_cache_1:.3f}s | Action 2 (dari RAM): {t_cache_2:.3f}s")

SOAL 2a: HASIL PARSING & DISTRIBUSI PARTISI DI WORKER
Total baris valid      : 119372
Total baris rusak (Acc): 0
Jumlah partisi         : 8
Sebaran per partisi    : [14336, 15360, 14336, 15360, 15360, 14336, 15360, 14924]

[Tanpa Cache] Action 1: 2.715s | Action 2: 2.713s
[Dengan Cache] Action 1 (materialisasi): 12.598s | Action 2 (dari RAM): 4.504s


In [3]:
# CELL 3: SOAL 2b - Tiga Analisis Utama RDD (Narrow vs Wide Transformations)
# (i) 10 wiki dengan edit manusia (type=edit, bot=false) terbanyak
top10_human_wiki_rdd = (
    parsed_rdd
    .filter(lambda r: r.get('type') == 'edit' and r.get('bot') is False)  # NARROW
    .map(lambda r: (r.get('wiki'), 1))                                    # NARROW
    .reduceByKey(lambda a, b: a + b)                                      # WIDE (Shuffle)
)
top10_human_wiki = top10_human_wiki_rdd.takeOrdered(10, key=lambda x: -x[1])
print("=== 2b (i) Top 10 Wiki Edit Manusia (reduceByKey) ===")
for wiki, cnt in top10_human_wiki:
    print(f"  - {wiki:<20}: {cnt}")

# (ii) 10 halaman paling banyak disunting pada idwiki
top10_idwiki_pages = (
    parsed_rdd
    .filter(lambda r: r.get('wiki') == 'idwiki' and r.get('type') == 'edit')  # NARROW
    .map(lambda r: (r.get('title'), 1))                                       # NARROW
    .reduceByKey(lambda a, b: a + b)                                          # WIDE (Shuffle)
    .takeOrdered(10, key=lambda x: -x[1])
)
print("\n=== 2b (ii) Top 10 Halaman Paling Banyak Disunting di idwiki ===")
for title, cnt in top10_idwiki_pages:
    print(f"  - {title:<40}: {cnt}")

# (iii) 15 kata terbanyak pada comment penyuntingan idwiki setelah stopword dibuang
STOPWORDS_ID_EN = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan', 'ini', 'itu',
    'the', 'to', 'of', 'and', 'in', 'a', 'is', 'for', 'on', 'with', 'as', 'by',
    'wikipedia', 'halaman', 'suntingan', 'bagian', 'pranala', 'kategori'
}

def tokenize_comment(comment):
    if not comment:
        return []
    cleaned = re.sub(r'/\*.*?\*/', ' ', comment.lower())
    words = re.findall(r'[a-z]{3,}', cleaned)
    return [w for w in words if w not in STOPWORDS_ID_EN]

top15_comment_words = (
    parsed_rdd
    .filter(lambda r: r.get('wiki') == 'idwiki')                # NARROW
    .flatMap(lambda r: tokenize_comment(r.get('comment', '')))  # NARROW
    .map(lambda w: (w, 1))                                      # NARROW
    .reduceByKey(lambda a, b: a + b)                            # WIDE (Shuffle)
    .takeOrdered(15, key=lambda x: -x[1])
)
print("\n=== 2b (iii) Top 15 Kata Comment idwiki (Tanpa Stopword) ===")
for word, cnt in top15_comment_words:
    print(f"  - {word:<20}: {cnt}")

=== 2b (i) Top 10 Wiki Edit Manusia (reduceByKey) ===
  - wikidatawiki        : 24705
  - enwiki              : 4669
  - commonswiki         : 2832
  - dewiki              : 659
  - frwiki              : 491
  - zhwiki              : 475
  - jawiki              : 429
  - eswiki              : 390
  - ruwiki              : 356
  - ptwiki              : 262

=== 2b (ii) Top 10 Halaman Paling Banyak Disunting di idwiki ===
  - Benny K. Harman                         : 9
  - Pembicaraan Pengguna:~2026-58660-35/Bak pasir: 4
  - Wikipedia:Bak pasir                     : 4
  - Lumbantungkup                           : 4
  - Al-Farazdaq                             : 4
  - Pengguna:YTTA12/bak pasir               : 3
  - Pembicaraan:Surat Tanda Nomor Kendaraan : 2
  - Tradisi Suci                            : 2
  - Ros Serey Sothea                        : 2
  - Templat:SNExamples1                     : 2

=== 2b (iii) Top 15 Kata Comment idwiki (Tanpa Stopword) ===
  - ditambahkan         : 173

In [4]:
# CELL 4: SOAL 2c & 2d (Bagian 2) - Benchmark reduceByKey vs groupByKey & Lineage RDD
t0 = time.time()
res_reduce = (
    parsed_rdd
    .filter(lambda r: r.get('type') == 'edit' and r.get('bot') is False)
    .map(lambda r: (r.get('wiki'), 1))
    .reduceByKey(lambda a, b: a + b)
    .takeOrdered(10, key=lambda x: -x[1])
)
time_reduce = time.time() - t0

t0 = time.time()
res_group = (
    parsed_rdd
    .filter(lambda r: r.get('type') == 'edit' and r.get('bot') is False)
    .map(lambda r: (r.get('wiki'), 1))
    .groupByKey()
    .mapValues(sum)
    .takeOrdered(10, key=lambda x: -x[1])
)
time_group = time.time() - t0

print("=" * 65)
print(f"SOAL 2c BENCHMARK: reduceByKey = {time_reduce:.4f}s vs groupByKey = {time_group:.4f}s")
print("-> Buka App UI (:4040) -> Tab Stages untuk screenshot Shuffle Write/Read!")
print("=" * 65)

print("\nSOAL 2d LINEAGE RDD (toDebugString):")
print(top10_human_wiki_rdd.toDebugString().decode('utf-8'))

SOAL 2c BENCHMARK: reduceByKey = 6.5216s vs groupByKey = 6.9430s
-> Buka App UI (:4040) -> Tab Stages untuk screenshot Shuffle Write/Read!

SOAL 2d LINEAGE RDD (toDebugString):
(8) PythonRDD[33] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[11] at mapPartitions at PythonRDD.scala:171 []
 |  ShuffledRDD[10] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(8) PairwiseRDD[9] at reduceByKey at C:\Users\ASUS\AppData\Local\Temp\ipykernel_25024\1193817841.py:7 []
    |  PythonRDD[8] at reduceByKey at C:\Users\ASUS\AppData\Local\Temp\ipykernel_25024\1193817841.py:7 []
    |  PythonRDD[5] at RDD at PythonRDD.scala:59 []
    |      CachedPartitions: 8; MemorySize: 10.7 MiB; DiskSize: 0.0 B
    |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:300 []


In [2]:
# CELL 5: SOAL 3a - DataFrame dengan Skema Eksplisit, Timestamp, selisih_byte & Temp View
wiki_schema = StructType([
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("title", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("timestamp", LongType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True),
    StructField("comment", StringType(), True)
])

df_raw = spark.read.schema(wiki_schema).json("C:/BigData/wikipedia_snapshot.jsonl")

df_clean = (
    df_raw
    .filter(F.col("wiki").isNotNull())
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("timestamp"))))
    .withColumn(
        "selisih_byte",
        F.coalesce(F.col("length.new"), F.lit(0)) - F.coalesce(F.col("length.old"), F.lit(0))
    )
)

df_clean.createOrReplaceTempView("wiki_edits")
df_clean.cache()
print("Skema DataFrame & Temporary View 'wiki_edits' siap:")
df_clean.printSchema()

Skema DataFrame & Temporary View 'wiki_edits' siap:
root
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: integer (nullable = true)
 |    |-- new: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- selisih_byte: integer (nullable = false)



In [6]:
# CELL 6: SOAL 3b - Empat Kueri Analitik Spark SQL
print("=== 3b (i) Jumlah Edit per Menit pada 5 Wiki Teratas ===")
spark.sql("""
    WITH top5_wiki AS (
        SELECT wiki
        FROM wiki_edits
        WHERE type = 'edit'
        GROUP BY wiki
        ORDER BY COUNT(*) DESC
        LIMIT 5
    )
    SELECT
        w.wiki,
        DATE_TRUNC('minute', w.event_time) AS menit,
        COUNT(*) AS jumlah_edit
    FROM wiki_edits w
    JOIN top5_wiki t ON w.wiki = t.wiki
    WHERE w.type = 'edit'
    GROUP BY w.wiki, DATE_TRUNC('minute', w.event_time)
    ORDER BY w.wiki, menit
""").show(15, truncate=False)

print("=== 3b (ii) Persentase Bot dan Manusia per Wiki (Konsisten dengan Soal 2) ===")
spark.sql("""
    SELECT
        wiki,
        COUNT(*) AS total_aktivitas,
        SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) AS jml_bot,
        SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) AS jml_manusia,
        ROUND(SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS persen_bot,
        ROUND(SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS persen_manusia
    FROM wiki_edits
    GROUP BY wiki
    ORDER BY total_aktivitas DESC
    LIMIT 10
""").show(truncate=False)

print("=== 3b (iii) Rata-rata dan Median (percentile_approx) abs(selisih_byte) per Type ===")
spark.sql("""
    SELECT
        type,
        COUNT(*) AS jumlah_event,
        ROUND(AVG(ABS(selisih_byte)), 2) AS rata_rata_abs_byte,
        PERCENTILE_APPROX(ABS(selisih_byte), 0.5) AS median_abs_byte
    FROM wiki_edits
    GROUP BY type
    ORDER BY jumlah_event DESC
""").show(truncate=False)

print("=== 3b (iv) 10 Halaman idwiki dengan Jumlah Editor Unik Terbanyak ===")
spark.sql("""
    SELECT
        title,
        COUNT(DISTINCT user) AS editor_unik,
        COUNT(*) AS total_edit
    FROM wiki_edits
    WHERE wiki = 'idwiki' AND type = 'edit'
    GROUP BY title
    ORDER BY editor_unik DESC, total_edit DESC
    LIMIT 10
""").show(truncate=False)

=== 3b (i) Jumlah Edit per Menit pada 5 Wiki Teratas ===
+-----------+-------------------+-----------+
|wiki       |menit              |jumlah_edit|
+-----------+-------------------+-----------+
|commonswiki|2026-11-19 11:44:00|45         |
|commonswiki|2026-11-19 11:55:00|167        |
|commonswiki|2026-11-19 11:56:00|482        |
|commonswiki|2026-11-19 11:57:00|420        |
|commonswiki|2026-11-19 11:58:00|436        |
|commonswiki|2026-11-19 11:59:00|227        |
|commonswiki|2026-11-19 12:00:00|79         |
|commonswiki|2026-11-19 12:01:00|230        |
|commonswiki|2026-11-19 12:02:00|235        |
|commonswiki|2026-11-19 12:03:00|251        |
|commonswiki|2026-11-19 12:04:00|275        |
|commonswiki|2026-11-19 12:05:00|308        |
|commonswiki|2026-11-19 12:06:00|303        |
|commonswiki|2026-11-19 12:07:00|226        |
|commonswiki|2026-11-19 12:08:00|96         |
+-----------+-------------------+-----------+
only showing top 15 rows
=== 3b (ii) Persentase Bot dan Manusia per W

In [7]:
# CELL 7: SOAL 3c - Deteksi Lonjakan Penyuntingan (Window 5 Menit & Persentil ke-99)
windowed_edits = (
    df_clean
    .filter(F.col("type") == "edit")
    .groupBy(
        F.window(F.col("event_time"), "5 minutes"),
        F.col("wiki"),
        F.col("title")
    )
    .agg(
        F.count("*").alias("jumlah_edit"),
        F.countDistinct("user").alias("editor_unik")
    )
)

threshold_k = windowed_edits.approxQuantile("jumlah_edit", [0.99], 0.01)[0]
print(f"=== SOAL 3c: Deteksi Lonjakan (Ambang Batas k pada Persentil ke-99 = {threshold_k}) ===")

anomalies = (
    windowed_edits
    .filter(F.col("jumlah_edit") >= threshold_k)
    .orderBy(F.desc("jumlah_edit"))
)
anomalies.show(15, truncate=False)

=== SOAL 3c: Deteksi Lonjakan (Ambang Batas k pada Persentil ke-99 = 114.0) ===
+------------------------------------------+------------+----------+-----------+-----------+
|window                                    |wiki        |title     |jumlah_edit|editor_unik|
+------------------------------------------+------------+----------+-----------+-----------+
|{2026-11-19 12:10:00, 2026-11-19 12:15:00}|wikidatawiki|Q140929546|114        |1          |
+------------------------------------------+------------+----------+-----------+-----------+



In [8]:
# CELL 8: SOAL 3d - Execution Plan (explain) & Optimasi spark.sql.shuffle.partitions
query_agg = spark.sql("""
    SELECT wiki, type, COUNT(*) AS cnt, AVG(selisih_byte) AS avg_diff
    FROM wiki_edits
    GROUP BY wiki, type
""")

print("=== SOAL 3d: Execution Plan (explain) ===")
query_agg.explain()

# Benchmark Default (200 shuffle partitions)
spark.conf.set("spark.sql.shuffle.partitions", "200")
t0 = time.time()
query_agg.collect()
t_default = time.time() - t0

# Benchmark Setelah Optimasi (8 shuffle partitions)
spark.conf.set("spark.sql.shuffle.partitions", "8")
t0 = time.time()
query_agg.collect()
t_opt = time.time() - t0

print(f"\n[Optimasi Shuffle Partitions] Default (200): {t_default:.4f}s | Optimasi (8): {t_opt:.4f}s")

=== SOAL 3d: Execution Plan (explain) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[wiki#2, type#1], functions=[count(1), avg(selisih_byte#14)])
   +- Exchange hashpartitioning(wiki#2, type#1, 40), ENSURE_REQUIREMENTS, [plan_id=1015]
      +- HashAggregate(keys=[wiki#2, type#1], functions=[partial_count(1), partial_avg(selisih_byte#14)])
         +- InMemoryTableScan [type#1, wiki#2, selisih_byte#14]
               +- InMemoryRelation [id#0L, type#1, wiki#2, server_name#3, title#4, namespace#5, user#6, bot#7, minor#8, timestamp#9L, length#10, comment#11, event_time#13, selisih_byte#14], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- *(1) Project [id#0L, type#1, wiki#2, server_name#3, title#4, namespace#5, user#6, bot#7, minor#8, timestamp#9L, length#10, comment#11, cast(from_unixtime(timestamp#9L, yyyy-MM-dd HH:mm:ss, Some(Asia/Jakarta)) as timestamp) AS event_time#13, (coalesce(length#10.new, 0) - coalesce(length#10.ol

---
### Bagian Akhir: Soal 4 (Spark Structured Streaming)
**Penting Sebelum Menjalankan Cell 9:**
Pastikan **Cell 3 (`run_producer()`) pada Notebook 1 (`01_MASTER_Kafka_Ingestion_dan_Export.ipynb`)** sedang menyala agar ada aliran data *real-time* masuk ke Kafka!
- Untuk **Soal 4b**, jalankan pertama kali dengan `TRIGGER_INTERVAL = "10 seconds"` selama ±3-4 menit, catat metrik `lastProgress`, lalu hentikan sel (`■`), ganti menjadi `"30 seconds"`, dan jalankan kembali.
- Untuk **Soal 4c**, hentikan sel (`■`) lalu jalankan ulang **tanpa menghapus** folder `Z:/checkpoints/wiki_stream_cp` untuk membuktikan `Batch ID` melanjutkan hitungan sebelumnya tanpa duplikasi.

In [4]:
# CELL 9: SOAL 4a, 4b & 4c - Spark Structured Streaming (Update Mode)
import time
import pyspark.sql.functions as F

spark.conf.set("spark.sql.shuffle.partitions", "4")

# Ganti menjadi "30 seconds" saat melakukan pengujian perbandingan Soal 4b
TRIGGER_INTERVAL = "30 seconds"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "192.168.110.109:9092")
    .option("subscribe", "wikipedia-recentchange")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), wiki_schema).alias("data"))
    .select("data.*")
    .filter(F.col("wiki").isNotNull() & F.col("timestamp").isNotNull())
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("timestamp"))))
)

windowed_metrics = (
    parsed_stream
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        F.window(F.col("event_time"), "1 minute"),
        F.col("wiki")
    )
    .agg(
        F.count("*").alias("total_event"),
        F.sum(F.when(F.col("bot") == True, 1).otherwise(0)).alias("bot_count"),
        F.round(
            F.sum(F.when(F.col("bot") == True, 1).otherwise(0)) / F.count("*"),
            3
        ).alias("rasio_bot")
    )
)

def write_to_dashboard_sink(batch_df, batch_id):
    row_count = batch_df.count()
    if row_count > 0:
        flat_df = batch_df.select(
            F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "wiki",
            "total_event",
            "bot_count",
            "rasio_bot"
        )
        flat_df.write.mode("append").parquet("./dashboard_metrics/")
        print(f"--- Batch {batch_id} ({row_count} baris diproses) ---")
        flat_df.orderBy(F.desc("total_event")).show(5, truncate=False)

query = (
    windowed_metrics.writeStream
    # DIUBAH KE "update" AGAR OUTPUT LANGSUNG KELUAR TIAP BATCH
    .outputMode("update") 
    .foreachBatch(write_to_dashboard_sink)
    # NAMA FOLDER CHECKPOINT DIUBAH AGAR TIDAK BENTROK DENGAN MODE APPEND SEBELUMNYA
    .option("checkpointLocation", "./checkpoints/wiki_stream_cp_update")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

# Monitoring metrik kinerja via lastProgress (Soal 4b)
# Monitoring metrik kinerja (Versi Debugging)
try:
    while query.isActive:
        # Panggil status internal Spark untuk melihat apa yang sedang dikerjakan
        current_status = query.status
        print(f"[{time.strftime('%X')}] Status Streaming: {current_status['message']}")
        
        progress = query.lastProgress
        if progress:
            print("\n=== TELEMETRI STREAMING (lastProgress) ===")
            print(f"Trigger Interval        : {TRIGGER_INTERVAL}")
            print(f"Batch ID                : {progress.get('batchId')}")
            print(f"Num Input Rows          : {progress.get('numInputRows')}")
            print(f"Input Rows / Sec        : {progress.get('inputRowsPerSecond')}")
            print(f"Processed Rows / Sec    : {progress.get('processedRowsPerSecond')}")
            print("-" * 40)
            
        time.sleep(10)
except KeyboardInterrupt:
    print("\nMenghentikan query streaming...")
    query.stop()

[14:08:27] Status Streaming: Initializing sources
[14:08:37] Status Streaming: Getting offsets from KafkaV2[Subscribe[wikipedia-recentchange]]
[14:08:47] Status Streaming: Writing offsets to log
--- Batch 18 (18 baris diproses) ---
+-------------------+-------------------+------------+-----------+---------+---------+
|window_start       |window_end         |wiki        |total_event|bot_count|rasio_bot|
+-------------------+-------------------+------------+-----------+---------+---------+
|2026-11-20 14:02:00|2026-11-20 14:03:00|commonswiki |272        |182      |0.669    |
|2026-11-20 14:02:00|2026-11-20 14:03:00|wikidatawiki|236        |0        |0.0      |
|2026-11-20 14:02:00|2026-11-20 14:03:00|enwiki      |165        |12       |0.073    |
|2026-11-20 14:02:00|2026-11-20 14:03:00|enwiktionary|133        |74       |0.556    |
|2026-11-20 14:02:00|2026-11-20 14:03:00|zhwikisource|73         |73       |1.0      |
+-------------------+-------------------+------------+-----------+------